[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C24_Inference_Serving_Course/02_continuous_batching/02_continuous_batching.ipynb)

# 02 · Continuous Batching（用 numpy 做调度器+账本）

目标：把 **静态批处理的气泡**、**迭代级连续批处理调度器**、**吞吐/延迟（TTFT/TPOT）账本**、**chunked prefill**、**抢占** 用 numpy 实现出来，并用 `assert` 钉死核心不变量。

路线：静态 batching 模拟器 → 连续 batching 调度器 → 吞吐对比 → TTFT/TPOT 与 P50/P99 → chunked prefill → 抢占（不死锁） → ✏️ 练习 → 📖 答案 → 🧪 真实速率胶囊。

> 心智模型：**调度器 = 一个事件循环；每个迭代 = 一次前向 = batch 里每个活跃请求产 1 个 token；连续批处理 = 完成即让位、到达即补位**。我们写的是*机制与正确性*，不是真实 tokens/s。

## 1 · 静态批处理模拟器：量化气泡

静态批处理整批同步推进，必须等 **批内最长** 的请求生成完。
用「迭代步」为时间单位（每步 = batch 里每个还在跑的请求各产 1 token）模拟：算总步数、有效 token、GPU 利用率（气泡）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def static_batch_run(gen_lengths):
    '''静态批处理：一批请求一起跑，整批跑 max(L) 步。
       返回 (总步数, 有效token数, batch内可计算槽总数, 利用率).'''
    B = len(gen_lengths)
    steps = max(gen_lengths)                 # 等最长的请求
    useful = sum(gen_lengths)                # 真正产出的 token
    slot_steps = B * steps                   # batch 槽 × 步数 = 理论可算的格子
    util = useful / slot_steps
    return steps, useful, slot_steps, util

gen = [10, 20, 6, 8, 15]                     # 5 个请求的生成长度（方差大）
steps, useful, slots, util = static_batch_run(gen)
print(f'静态: {len(gen)} 请求, 总步数={steps}, 有效token={useful}, 槽×步={slots}, 利用率={util:.0%}')
assert steps == 20, '总步数 = max(L)'
assert useful == sum(gen)
assert util == sum(gen) / (len(gen) * max(gen))
assert util < 0.65, '长度方差大 -> 大量气泡'
print('✅ 静态批处理利用率 < 65%：短请求停了仍占槽，GPU 为它们空转（气泡）')

## 2 · 连续批处理调度器：完成即让位、到达即补位

迭代级调度：每步前向后，把完成的请求移出、从等待队列补入新请求填满 batch。
请求用 `(到达步, 生成长度)` 描述。我们跑一个事件循环，记录每个请求的开始/完成步，并统计总步数。

In [ ]:
def continuous_batch_run(requests, max_batch):
    '''requests: list of (arrival_step, gen_len). 迭代级调度。
       返回 dict: total_steps, finish(每请求完成步), start(开始生成步).'''
    reqs = sorted(range(len(requests)), key=lambda i: requests[i][0])  # 按到达排序
    waiting = list(reqs)
    running = {}                              # rid -> 剩余要生成的 token 数
    start, finish = {}, {}
    step = 0
    while waiting or running:
        # 纳新：到达且 batch 没满的请求加入 running
        still = []
        for rid in waiting:
            arr, gl = requests[rid]
            if arr <= step and len(running) < max_batch:
                running[rid] = gl; start[rid] = step
            else:
                still.append(rid)
        waiting = still
        if not running:                       # 没人能跑（都还没到达）-> 时间前进
            step += 1; continue
        # 前向一步：每个 running 请求产 1 token
        for rid in list(running):
            running[rid] -= 1
            if running[rid] == 0:             # 完成 -> 退役、让位
                finish[rid] = step + 1
                del running[rid]
        step += 1
    return dict(total_steps=step, finish=finish, start=start)

# 5 个请求都在第 0 步到达，batch 容量 5
requests = [(0, 10), (0, 20), (0, 6), (0, 8), (0, 15)]
res = continuous_batch_run(requests, max_batch=5)
print('总步数 =', res['total_steps'], '  各请求完成步 =', dict(sorted(res['finish'].items())))
assert res['total_steps'] == 20, '5 个并发<=batch时, 总步数仍是 max(L)=20'
assert res['finish'][2] == 6, '最短请求(6 tok)第 6 步就完成并让位（不必等到 20）'
print('✅ 连续批处理：短请求第 6 步就完成退役，槽位立刻可让给新请求')

## 3 · 吞吐对比：连续 vs 静态（同一批请求）

真正的差距在 **新请求不断到达** 时显现：静态批处理要等整批结束才接下一批，连续批处理随到随补。
用一个「源源不断到达」的负载，对比两者处理完所有请求的总步数与吞吐（token/步）。

In [ ]:
def static_throughput(requests, max_batch):
    '''静态：把请求按到达切成一批批（每批 max_batch 个），每批跑 max(L) 步，批间不重叠。'''
    order = sorted(range(len(requests)), key=lambda i: requests[i][0])
    total_steps = 0
    for b in range(0, len(order), max_batch):
        batch = order[b:b+max_batch]
        # 这一批要等其中最晚到达者才能开始，再跑 max(L) 步
        arrivals = [requests[i][0] for i in batch]
        lengths = [requests[i][1] for i in batch]
        start = max(total_steps, max(arrivals))
        total_steps = start + max(lengths)
    return total_steps

# 12 个请求，错峰到达，生成长度方差大
rng2 = np.random.default_rng(1)
N = 12
arr = np.cumsum(rng2.integers(0, 3, size=N))           # 递增到达步
lens = rng2.integers(4, 25, size=N)
loadreq = [(int(a), int(l)) for a, l in zip(arr, lens)]

s_steps = static_throughput(loadreq, max_batch=4)
c_steps = continuous_batch_run(loadreq, max_batch=4)['total_steps']
total_tok = int(lens.sum())
print(f'总产出 token = {total_tok}')
print(f'静态:   {s_steps} 步, 吞吐 = {total_tok/s_steps:.2f} tok/步')
print(f'连续:   {c_steps} 步, 吞吐 = {total_tok/c_steps:.2f} tok/步')
assert c_steps <= s_steps, '连续批处理总步数不应多于静态'
assert total_tok/c_steps >= total_tok/s_steps, '连续吞吐 >= 静态'
print(f'✅ 连续批处理更快 {s_steps-c_steps} 步、吞吐更高 —— 消灭了批间等待与队内气泡')

## 4 · 延迟账本：TTFT / TPOT 与 P50 / P99

**TTFT**（首 token 延迟）= 开始生成步 − 到达步（含排队）；**TPOT**（每 token 延迟）= 生成阶段平均每 token 步数。
服务 SLO 通常约束 **尾延迟**（P99），因为它决定最差用户体验。我们从连续批处理的结果里算这些指标。

In [ ]:
def latency_metrics(requests, res):
    '''从调度结果算每个请求的 TTFT 与 TPOT（以步为单位）。'''
    ttft, tpot = [], []
    for rid, (arr, gl) in enumerate(requests):
        first_tok_step = res['start'][rid]            # 开始生成（这里简化：prefill 后第一步）
        ttft.append(first_tok_step - arr)             # 含排队等待
        gen_span = res['finish'][rid] - res['start'][rid]
        tpot.append(gen_span / gl)                    # 平均每 token 的步数
    return np.array(ttft), np.array(tpot)

def pctl(x, q):
    return float(np.percentile(x, q))

res = continuous_batch_run(loadreq, max_batch=4)
ttft, tpot = latency_metrics(loadreq, res)
print(f'TTFT  P50={pctl(ttft,50):.1f}  P99={pctl(ttft,99):.1f} 步')
print(f'TPOT  P50={pctl(tpot,50):.2f}  P99={pctl(tpot,99):.2f} 步/token')
assert (ttft >= 0).all(), 'TTFT 不应为负'
assert pctl(ttft,99) >= pctl(ttft,50), 'P99 >= P50（尾延迟更高）'
print('✅ 算出 TTFT/TPOT 的 P50/P99 —— SLO 通常卡 P99 尾延迟')

## 5 · Chunked Prefill：长 prefill 切碎喂

一个长 prompt 的 prefill 若一次做完会噎住所有 decode。chunked prefill 给每个迭代设 **token 预算**，
先排 decode（每个占 1 token），剩余预算做 prefill 的一个分块。

**不变量**：所有分块的 prefill token 加起来 = prompt 长度；每步总 token ≤ 预算。

In [ ]:
def chunked_prefill(prompt_len, num_decode, budget):
    '''把 prompt_len 的 prefill 切块，与 num_decode 个 decode 请求混跑。
       返回每个迭代的 (decode_tokens, prefill_tokens).'''
    assert budget > num_decode, 'token 预算必须能容下所有 decode + 至少 1 prefill'
    remaining = prompt_len
    schedule = []
    while remaining > 0:
        chunk = min(remaining, budget - num_decode)   # 这一步能塞多少 prefill
        schedule.append((num_decode, chunk))
        remaining -= chunk
    return schedule

prompt_len, num_decode, budget = 1000, 30, 256
sched = chunked_prefill(prompt_len, num_decode, budget)
print(f'prompt={prompt_len}, decode={num_decode}, 预算={budget} -> 切成 {len(sched)} 块')
print('前 3 个迭代 (decode, prefill):', sched[:3])
# 不变量 1：prefill 总量精确等于 prompt 长度
assert sum(pf for _, pf in sched) == prompt_len, 'prefill token 总量必须等于 prompt 长度'
# 不变量 2：每步总 token 不超预算
assert all(dc + pf <= budget for dc, pf in sched), '每步总 token 不能超预算'
print('✅ chunked prefill：prefill 被摊到多步，每步不超预算 —— 削平 TPOT 尖峰（总量不变）')

## 6 · 抢占：显存压力下系统仍能推进（不死锁）

当 KV 块不够让所有 running 请求继续时，抢占一些请求（这里用 recompute：丢 KV、降回等待）。
**核心不变量**：即使所有请求的总 KV 需求远超显存，系统也能逐个把它们跑完、**不死锁**。

In [ ]:
def run_with_preemption(gen_lengths, total_blocks, block_size, max_batch):
    '''每请求需 ceil(已生成/block_size) 块。显存不足时抢占最晚加入的请求（recompute）。
       返回 (完成的请求数, 总步数)。断言所有请求最终都完成。'''
    n = len(gen_lengths)
    waiting = list(range(n))
    progress = {i: 0 for i in range(n)}       # 已生成 token（被抢占则清零=recompute）
    running = []                              # rid 列表，保持加入顺序
    done = set()
    step = 0
    def blocks_needed(rid):
        return max(1, (progress[rid] + block_size - 1) // block_size)
    while len(done) < n:
        # 纳新
        for rid in list(waiting):
            if len(running) < max_batch:
                running.append(rid); waiting.remove(rid)
        # 显存检查：running 总块需求超显存 -> 抢占最晚加入者
        while sum(blocks_needed(r) for r in running) > total_blocks and len(running) > 1:
            victim = running.pop()            # 最晚加入的被抢占
            progress[victim] = 0             # recompute：丢弃 KV
            waiting.append(victim)
        # 前向一步
        for rid in list(running):
            progress[rid] += 1
            if progress[rid] >= gen_lengths[rid]:
                done.add(rid); running.remove(rid)
        step += 1
        if step > 100000:                     # 安全阀：若死锁会卡住
            break
    return len(done), step

# 故意让显存很紧：4 个请求，每个最终要好几块，但显存只有 3 块
gen_lengths = [9, 12, 7, 10]
done, steps = run_with_preemption(gen_lengths, total_blocks=3, block_size=4, max_batch=4)
print(f'显存仅 3 块, 4 个请求 -> 完成 {done}/4, 用了 {steps} 步')
assert done == len(gen_lengths), '抢占必须保证所有请求最终完成（不死锁/不饿死）'
print('✅ 即使显存远不够同时装下所有请求，抢占让系统逐个推进、全部完成 —— 不死锁')

---
## ✏️ 练习 1：调度器的一步 `schedule_step`

实现连续批处理的**单步**逻辑 `schedule_step(running, waiting, max_batch)`：
`running` 是 dict `{rid: 剩余token}`，`waiting` 是待入队 rid 的 list（已到达）。
(a) 先从 `waiting` 头部补入请求直到 `running` 满或 `waiting` 空（初始 token 数从 `gen` 全局 dict 取）；
(b) 对每个 `running` 请求剩余 token −1，归零的移出并收集到 `finished`；返回 `(running, waiting, finished)`。

In [ ]:
def schedule_step(running, waiting, max_batch):
    # 约定：全局 dict gen[rid] = 该请求的生成长度（自测 cell 里已定义）
    # TODO:
    #  1) 当 len(running) < max_batch 且 waiting 非空：pop(0) 一个 rid 加入 running，
    #     初始剩余 token = gen[rid]
    #  2) 每个 running 请求剩余 -1；==0 的移出并加入 finished 列表
    #  3) 返回 (running, waiting, finished)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
gen = {0:5, 1:3, 2:8, 3:2, 4:6}              # 全局：rid -> 生成长度
running = {}
waiting = [0, 1, 2, 3, 4]
running, waiting, fin = schedule_step(running, waiting, max_batch=2)
assert len(running) == 2, '应补到 batch 满（2）'
assert running == {0:4, 1:2}, '补入 0,1 并各 -1'
assert fin == [] and waiting == [2,3,4]
# 再跑两步，请求1(初始3->此刻2)应在某步完成
for _ in range(2):
    running, waiting, fin = schedule_step(running, waiting, max_batch=2)
assert 1 in fin or all(r != 1 for r in running), '请求1 应已完成并让位'
print('✅ 练习 1 通过：单步调度正确补位、推进、退役')

## ✏️ 练习 2：吞吐与利用率计算

实现 `efficiency(gen_lengths, total_steps)`：给一批请求的生成长度与某调度跑出的总步数，
返回 `(吞吐, 理想总步数, 效率)`：吞吐 = 总token/总步数；理想总步数 = 所有 token 串成一条但被 batch 并行后的下界 `max(单请求最长, ceil(总token/容量))` —— 这里简化为 `效率 = 理想/实际`，理想用 `max(gen_lengths)`（全并发下界）。

In [ ]:
def efficiency(gen_lengths, total_steps, ideal_steps=None):
    # TODO:
    #   throughput = sum(gen_lengths) / total_steps
    #   ideal = ideal_steps if 给定 else max(gen_lengths)  # 全并发下界
    #   eff = ideal / total_steps
    #   返回 (throughput, ideal, eff)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
gl = [10, 20, 6, 8, 15]
thr, ideal, eff = efficiency(gl, total_steps=20)   # 全并发<=batch 时 20 步
assert abs(thr - sum(gl)/20) < 1e-9
assert ideal == 20 and abs(eff - 1.0) < 1e-9, '全并发=下界 -> 效率 100%'
thr2, ideal2, eff2 = efficiency(gl, total_steps=40)
assert eff2 < eff, '总步数翻倍 -> 效率减半'
print(f'吞吐={thr:.2f} tok/步, 理想步={ideal}, 效率={eff:.0%}')
print('✅ 练习 2 通过：吞吐与效率计算正确')

## ✏️ 练习 3：SLO 达标率

实现 `slo_attainment(ttft, tpot, ttft_slo, tpot_slo)`：给每个请求的 TTFT/TPOT 数组与各自 SLO 上限，
返回 `(同时满足两个SLO的请求比例, 仅TTFT达标比例, 仅TPOT达标比例)`。这是 goodput 的基础。

In [ ]:
def slo_attainment(ttft, tpot, ttft_slo, tpot_slo):
    # TODO: 用 numpy 布尔运算：
    #   ok_ttft = ttft <= ttft_slo ; ok_tpot = tpot <= tpot_slo
    #   both = (ok_ttft & ok_tpot).mean()
    #   返回 (both, ok_ttft.mean(), ok_tpot.mean())
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
ttft = np.array([1.0, 2.0, 5.0, 0.5])
tpot = np.array([0.04, 0.06, 0.03, 0.05])
both, a, b = slo_attainment(ttft, tpot, ttft_slo=3.0, tpot_slo=0.05)
# TTFT<=3: [T,T,F,T]=3/4 ; TPOT<=0.05: [T,F,T,T]=3/4 ; both: [T,F,F,T]=2/4
assert abs(a - 0.75) < 1e-9 and abs(b - 0.75) < 1e-9
assert abs(both - 0.5) < 1e-9, '同时达标的只有 2/4'
print(f'同时达标={both:.0%}, 仅TTFT={a:.0%}, 仅TPOT={b:.0%}')
print('✅ 练习 3 通过：SLO 达标率是 goodput 的基础')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def schedule_step(running, waiting, max_batch):
    finished = []
    while len(running) < max_batch and waiting:
        rid = waiting.pop(0)
        running[rid] = gen[rid]
    for rid in list(running):
        running[rid] -= 1
        if running[rid] == 0:
            finished.append(rid)
            del running[rid]
    return running, waiting, finished

In [ ]:
# 练习 2 参考答案
def efficiency(gen_lengths, total_steps, ideal_steps=None):
    throughput = sum(gen_lengths) / total_steps
    ideal = ideal_steps if ideal_steps is not None else max(gen_lengths)
    eff = ideal / total_steps
    return throughput, ideal, eff

In [ ]:
# 练习 3 参考答案
def slo_attainment(ttft, tpot, ttft_slo, tpot_slo):
    ok_ttft = ttft <= ttft_slo
    ok_tpot = tpot <= tpot_slo
    both = float((ok_ttft & ok_tpot).mean())
    return both, float(ok_ttft.mean()), float(ok_tpot.mean())

---
## 🧪 真实数据胶囊：用真实 prefill/decode 速率算吞吐与延迟

下面是几个**真实量级**的服务速率（公开 benchmark 的近似值：单卡上 Llama-3-8B 量级，prefill 远快于 decode 的 per-token 速率，因为 prefill 并行处理整段 prompt）。
用它们把「迭代步」换算成真实时间，算 TTFT 与 TPOT，把纸面调度接到真实数字。

> 联网失败时回退到内置的真实量级常数，不影响运行。

In [ ]:
# 真实量级服务速率（约数，来自公开推理 benchmark；单卡 7-8B 模型）
def load_serving_rates():
    try:
        # 真实场景会从 benchmark 服务/文件读取；本环境无网络 -> 触发回退
        raise RuntimeError('offline')
    except Exception:
        return {
            'prefill_tok_per_s': 12000.0,   # prefill 吞吐：并行处理 prompt，很快
            'decode_tok_per_s_per_req': 80.0,  # 单请求 decode：~80 tok/s (TPOT~12.5ms)
        }

rates = load_serving_rates()
def ttft_seconds(prompt_len, rates):
    return prompt_len / rates['prefill_tok_per_s']
def tpot_seconds(rates):
    return 1.0 / rates['decode_tok_per_s_per_req']

for plen in [128, 1024, 4096]:
    print(f'prompt={plen:5d} -> TTFT ≈ {ttft_seconds(plen, rates)*1000:6.1f} ms, '
          f'TPOT ≈ {tpot_seconds(rates)*1000:.1f} ms/token')
assert ttft_seconds(4096, rates) > ttft_seconds(128, rates), 'prompt 越长 TTFT 越大'
print('\n观察：prefill 并行 -> TTFT 随 prompt 长增长；decode 串行 -> TPOT 固定。这就是两阶段延迟的来源。')

**🧪 胶囊练习**：实现 `request_latency(prompt_len, gen_len, rates)`：返回一个请求的**端到端延迟**（秒）
= TTFT（prefill 整段 prompt）+ 生成 `gen_len` 个 token 的时间（每个 TPOT）。用它说明长输出请求的延迟由 decode 主导。

In [ ]:
def request_latency(prompt_len, gen_len, rates):
    # TODO: ttft = prompt_len / prefill_tok_per_s ;
    #       decode_time = gen_len / decode_tok_per_s_per_req ;
    #       返回 ttft + decode_time
    raise NotImplementedError

In [ ]:
# 自测
rates = load_serving_rates()
lat = request_latency(prompt_len=1024, gen_len=200, rates=rates)
expected = 1024/12000.0 + 200/80.0
assert abs(lat - expected) < 1e-9
# 长输出时 decode 主导
lat_short = request_latency(2048, 10, rates)
lat_long  = request_latency(2048, 500, rates)
assert lat_long > lat_short and (500/80.0) > (2048/12000.0), '长输出 -> decode 主导延迟'
print(f'prompt=1024,gen=200 -> 端到端 {lat:.2f}s (TTFT {1024/12000*1000:.0f}ms + decode {200/80:.2f}s)')
print('✅ 胶囊练习通过：长输出请求的延迟由 decode（TPOT×token数）主导')

In [ ]:
# 📖 胶囊参考答案
def request_latency(prompt_len, gen_len, rates):
    ttft = prompt_len / rates['prefill_tok_per_s']
    decode_time = gen_len / rates['decode_tok_per_s_per_req']
    return ttft + decode_time

---
### 小结
- **静态批处理** 整批同步、等最长请求 → 队内气泡 + 队首阻塞，利用率常 20–60%。
- **连续批处理**（Orca）= 迭代级调度：完成即让位、到达即补位，几乎步步满载，吞吐数倍提升。
- 调度循环 = **纳新 → 前向一步 → 退役 → 处理压力**；vLLM 用 **waiting/running/swapped** 三队列驱动。
- **prefill/decode 干扰**：长 prefill 噎住 decode → **chunked prefill** 切碎喂（总量不变、每步不超预算）。
- **抢占** 是过载安全网：显存不够时换出/重算，保证系统推进**不死锁**（但要防饥饿）。
- 优化目标是 **goodput**（满足 SLO 的吞吐），不是裸吞吐；SLO 通常卡 P99 尾延迟。

下一站：**模块 03 · 投机解码** —— batch 填满了，怎么让 decode 一步确认多个 token、还不改变输出分布。